# Test polytopes — CNN

Analogous to `test_polytopes.ipynb` but for `FashionCNN_Small`.

## Libraries

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(ROOT)
print("Project root:", ROOT)

import torch

from src.models.networks import FashionCNN_Small
from src.quantization.quantize import quantize_model
from src.optim.build_polytopes import check_polytope_membership
from src.optim.build_polytopes_cnn import (
    build_two_class_cnn_polytopes,
    build_cnn_all_polytopes,
)

torch.manual_seed(42)

Project root: /Users/jeremiecabessa/Desktop/ROOT/Articles/Conference_Papers/2025_With_Jiri/code/ErrorVolumePolytopes


## Load model, qmodels, and dataset

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", device)

# Full-precision CNN
model = FashionCNN_Small()
model.load_state_dict(
    torch.load(os.path.join(ROOT, "checkpoints", "fashion_cnn_best.pth"),
               weights_only=True, map_location=device)
)
model.to(device).eval()

# Quantized models
BITS_GRID = [4, 6, 8, 16]
qmodels = {b: quantize_model(model, bits=b).eval() for b in BITS_GRID}

# Dataset (correctly-classified samples for the CNN)
dataset = torch.load(
    os.path.join(ROOT, "data", "fashionMNIST_correct_cnn.pt"),
    weights_only=False
)
print(f"Dataset size: {len(dataset)}")

# Quick sanity check on first sample
x0, c = dataset[0]
print(f"Sample shape : {x0.shape}  |  label : {c}")
print(f"Pixel range  : [{x0.min().item():.2f}, {x0.max().item():.2f}]")

Device: mps
Dataset size: 59297
Sample shape : torch.Size([1, 28, 28])  |  label : 9
Pixel range  : [-1.00, 1.00]


## Single sample — correct polytope

In [3]:
SAMPLE_IDX = 123
x0, c = dataset[SAMPLE_IDX]
x0 = x0.to(device)
x_in = x0.unsqueeze(0)   # (1, 1, 28, 28)
qmodel = qmodels[4]  # 4-bit quantized model for testing

print(f"Sample {SAMPLE_IDX}  |  label {c}")

A_correct, b_correct, _, _ = build_two_class_cnn_polytopes(model, qmodel, x_in, int(c))
print(f"\nA_correct shape : {tuple(A_correct.shape)}")
print(f"b_correct shape : {tuple(b_correct.shape)}")

# Membership test (cheap: just Ax + b <= 0)
x_vec = x0.flatten()
in_correct = check_polytope_membership(A_correct, b_correct, x_vec)
print(f"\nx_0 in correct polytope : {in_correct.item()}")

Sample 123  |  label 2

A_correct shape : (41361, 784)
b_correct shape : (41361,)

x_0 in correct polytope : True


## Single sample — b-approximated polytopes (all bit-widths)

In [4]:
A_correct2, b_correct2, polytopes_dict = build_cnn_all_polytopes(
    model, qmodels, x_in, int(c)
)

print(f"A_correct shape (build_cnn_all_polytopes) : {tuple(A_correct2.shape)}")
print()
for bits, (A_both, b_both) in polytopes_dict.items():
    in_both = check_polytope_membership(A_both, b_both, x_vec)
    print(f"  bits={bits:2d}  A_both shape={tuple(A_both.shape)}"
          f"  x_0 in both polytope: {in_both.item()}")

A_correct shape (build_cnn_all_polytopes) : (20685, 784)

  bits= 4  A_both shape=(41370, 784)  x_0 in both polytope: True
  bits= 6  A_both shape=(41370, 784)  x_0 in both polytope: True
  bits= 8  A_both shape=(41370, 784)  x_0 in both polytope: True
  bits=16  A_both shape=(41370, 784)  x_0 in both polytope: True


## Many samples — membership check

For each sample, verify:
- `x_0` is classified correctly by the full-precision model
- `x_0` lies inside the **correct polytope**
- `x_0` lies inside each **b-approximated polytope** (only when the qmodel also classifies correctly)

In [5]:
N = 10   # number of samples to test

stats = {
    "correct_polytope_ok": 0,
    **{f"both_polytope_ok_bits{b}": 0 for b in BITS_GRID},
    **{f"qmodel_correct_bits{b}": 0 for b in BITS_GRID},
}

with torch.no_grad():
    for i in range(N):
        xi, ci = dataset[i]
        xi = xi.to(device)
        x_in_i = xi.unsqueeze(0)
        x_vec_i = xi.flatten()

        # Build correct polytope
        A_c, b_c, _, _ = build_two_class_cnn_polytopes(model, qmodel, x_in_i, int(ci))
        ok_c = check_polytope_membership(A_c, b_c, x_vec_i).item()
        stats["correct_polytope_ok"] += int(ok_c)

        # Build all b-approximated polytopes
        _, _, pd = build_cnn_all_polytopes(model, qmodels, x_in_i, int(ci))
        for b, (A_b, b_b) in pd.items():
            # Check if qmodel also classifies correctly
            qpred = qmodels[b](x_in_i).argmax(dim=1).item()
            qok = int(qpred == ci)
            stats[f"qmodel_correct_bits{b}"] += qok

            ok_b = check_polytope_membership(A_b, b_b, x_vec_i).item()
            stats[f"both_polytope_ok_bits{b}"] += int(ok_b)

        print(f"Sample {i:3d}  correct_polytope={ok_c}", end="")
        for b in BITS_GRID:
            print(f"  bits{b}={int(check_polytope_membership(pd[b][0], pd[b][1], x_vec_i).item())}", end="")
        print()

print("\n--- Summary ---")
for k, v in stats.items():
    print(f"  {k}: {v} / {N}")

Sample   0  correct_polytope=True  bits4=1  bits6=1  bits8=1  bits16=1
Sample   1  correct_polytope=True  bits4=1  bits6=1  bits8=1  bits16=1
Sample   2  correct_polytope=True  bits4=1  bits6=1  bits8=1  bits16=1
Sample   3  correct_polytope=True  bits4=1  bits6=1  bits8=1  bits16=1
Sample   4  correct_polytope=True  bits4=1  bits6=1  bits8=1  bits16=1
Sample   5  correct_polytope=True  bits4=1  bits6=1  bits8=1  bits16=1
Sample   6  correct_polytope=True  bits4=1  bits6=1  bits8=1  bits16=1
Sample   7  correct_polytope=True  bits4=0  bits6=1  bits8=1  bits16=1
Sample   8  correct_polytope=True  bits4=1  bits6=1  bits8=1  bits16=1
Sample   9  correct_polytope=True  bits4=1  bits6=1  bits8=1  bits16=1

--- Summary ---
  correct_polytope_ok: 10 / 10
  both_polytope_ok_bits4: 9 / 10
  both_polytope_ok_bits6: 10 / 10
  both_polytope_ok_bits8: 10 / 10
  both_polytope_ok_bits16: 10 / 10
  qmodel_correct_bits4: 9 / 10
  qmodel_correct_bits6: 10 / 10
  qmodel_correct_bits8: 10 / 10
  qmodel_co

## Constraint structure analysis

Breakdown of constraint count by layer type to understand the LP scalability.

In [6]:
import numpy as np
import torch.nn as nn
from src.optim.build_polytopes_cnn import _Message, _Constraints, _collect, model_to_sequential

x0, c = dataset[0]
x_sample = x0.to(device)

# Collect constraints layer-by-layer
seq = model_to_sequential(model)
cst = _Constraints()
msg = _Message(x_sample)
with torch.no_grad():
    msg = _collect(seq, msg, cst)

print("Constraint breakdown per layer:")
print(f"  Unsaturated (ReLU active) layers : {len(cst.U_weight)}")
print(f"  Saturated   (ReLU / MaxPool)     : {len(cst.S_weight)}")
print()
total_U = sum(w.shape[0] for w in cst.U_weight)
total_S = sum(w.shape[0] for w in cst.S_weight)
print(f"  Total unsaturated constraints : {total_U}")
print(f"  Total saturated   constraints : {total_S}")
print(f"  Classification constraints   : 9  (n_classes - 1)")
print(f"  Grand total                  : {total_U + total_S + 9}")
print()

# Zero-norm rows (numerically uninformative)
A_c, b_c, _, _ = build_two_class_cnn_polytopes(model, qmodel, x0.unsqueeze(0).to(device), int(c))
A_np = A_c.cpu().numpy()
row_norms = np.linalg.norm(A_np, axis=1)
print(f"  Zero-norm rows (prunable)     : {(row_norms == 0).sum()}")
print(f"  Non-zero rows                 : {(row_norms > 0).sum()}")

Constraint breakdown per layer:
  Unsaturated (ReLU active) layers : 3
  Saturated   (ReLU / MaxPool)     : 5

  Total unsaturated constraints : 3578
  Total saturated   constraints : 17098
  Classification constraints   : 9  (n_classes - 1)
  Grand total                  : 20685

  Zero-norm rows (prunable)     : 8843
  Non-zero rows                 : 32518
